# PRAKTIKUM 07 — Ekspor Model dan Hasil Studi Kasus Preeklamsia (PE)

**Tema:** *model persistence, reproducibility, artifact integrity, model card, dan kesiapan inferensi aplikasi*  
**Jenis:** Praktikum Machine Learning — lanjutan Praktikum 05–06  
**Prasyarat:** Pandas, Pipeline, Decision Tree, train/test split, metrik evaluasi  
**Durasi anjuran:** 90–120 menit; latihan dapat dilanjutkan mandiri.

> **Batasan penggunaan:** Ini model pendidikan pada dataset studi kasus, bukan alat diagnosis/skrining klinis. Validitas definisi outcome, identitas pasien unik, urutan waktu prediktor, kalibrasi probabilitas, dan generalisasi eksternal **belum dibuktikan**. Jangan memasukkan identitas pasien ke tabel contoh atau log.

**Posisi dalam rangkaian:** Praktikum 05 menghasilkan data tabular bersih; Praktikum 06 memilih dan mengevaluasi model; Praktikum 07 **mempaketkan model terpilih yang sama dan memeriksa konsistensinya**; Praktikum 08 menggunakan artefak tersebut dalam aplikasi.

## A. Capaian pembelajaran dan indikator keberhasilan

Setelah menyelesaikan praktikum, mahasiswa mampu:

1. Menjelaskan mengapa **seluruh Pipeline** perlu disimpan, bukan hanya Decision Tree.
2. Membaca konfigurasi model dan metadata eksperimen dari Praktikum 06.
3. Melatih kembali **konfigurasi final hasil CV Modul 06** tanpa memakai test untuk memilih parameter.
4. Mengekspor dan memuat ulang model memakai `joblib` dari sumber tepercaya.
5. Membandingkan prediksi sebelum/sesudah *reload* dan dengan catatan evaluasi Modul 06.
6. Membuat **model card**, checksum SHA-256, dan manifest artefak.
7. Memeriksa kontrak delapan fitur, penanganan *missing values*, dan input baru.
8. Menjelaskan risiko *version mismatch*, *unsafe deserialization*, dan batas penggunaan klinis.

**Kriteria selesai:** notebook *Restart Kernel → Run All* tanpa error; dua salinan model identik; hasil prediksi 141 record test konsisten dengan Praktikum 06; metadata tersimpan, dapat dibaca ulang, dan berisi keterbatasan penggunaan.

## B. Teori singkat: apa yang sebenarnya disimpan?

`fit()` menghasilkan *state* terpelajar, bukan hanya hiperparameter. Pada rangkaian ini, state meliputi **median imputasi numerik**, **modus imputasi kategorikal**, **vocabulary kategori One-Hot Encoder**, dan **struktur Decision Tree**.

- `Pipeline` mengikat transformasi dan model dalam **satu objek**.
- `joblib.dump()` menyimpan objek Python terlatih; `joblib.load()` memuatnya kembali.
- `JSON` menyimpan metadata yang dapat dibaca manusia, **bukan** pengganti pipeline.
- `SHA-256` memeriksa integritas byte file (mendeteksi perubahan), **bukan** membuktikan file aman.
- Artefak bukan file yang independen dari lingkungan: kompatibilitas versi Python/scikit-learn perlu dicatat dan diuji.

**Penting:** `joblib.load()` berbasis mekanisme deserialisasi Python dan dapat mengeksekusi kode berbahaya dari file tidak tepercaya. Muat hanya artefak yang dibuat sendiri atau tervalidasi asalnya.

## C. Persiapan lingkungan dan berkas

**Struktur paket ZIP** (sudah tersedia tanpa unduhan internet):

```text
paket-praktikum07/
├── 07_Ekspor_Model_dan_Hasil_Modul_Final_dan_Jawaban.ipynb
├── studi_kasus/
│   ├── data/hasil/data_bersih.csv
│   ├── hasil/laporan_praktikum06.json
│   ├── hasil/metrik_perbandingan.csv
│   ├── hasil/prediksi_test.csv
│   └── hasil/ringkasan_cv.csv
└── deployment/versi_2_hasil_studi_kasus/app.py
```

Berkas dataset disiapkan dari hasil Praktikum 05, sementara catatan evaluasi dan hasil prediksi berasal dari Praktikum 06. Karena file *fitted model* tidak diekspor pada Modul 06, kita **merekonstruksi dan me-fit** pipeline dengan parameter yang telah dipilih di sana, kemudian memeriksa prediksi terhadap hasil Modul 06. Kita **tidak** melakukan pencarian hyperparameter atau pemilihan model baru pada Modul 07.

In [1]:
from pathlib import Path
import json, hashlib, platform, shutil, sys
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display

# Pastikan notebook dijalankan dari folder paket / subfoldernya.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'studi_kasus/hasil/laporan_praktikum06.json').exists()
             and (p / 'studi_kasus/data/hasil/data_bersih.csv').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Ekstrak paket Praktikum 07, lalu jalankan notebook dari folder tersebut.')

CASE = ROOT / 'studi_kasus'
RESULTS = CASE / 'hasil'
MODEL_DIR = CASE / 'model'
DEPLOY_DIR = ROOT / 'deployment/versi_2_hasil_studi_kasus'
DEPLOY_MODELS = DEPLOY_DIR / 'models'
DEPLOY_DATA = DEPLOY_DIR / 'data'
for p in [RESULTS, MODEL_DIR, DEPLOY_MODELS, DEPLOY_DATA]:
    p.mkdir(parents=True, exist_ok=True)

print('ROOT:', ROOT)
print('Python:', platform.python_version(), '| pandas:', pd.__version__,
      '| scikit-learn:', sklearn.__version__, '| joblib:', joblib.__version__)

ROOT: /Users/masaboe/Documents/NGAJAR/ngudiWaluyo2026/Materi/Workshop_From_Data_to_ML
Python: 3.10.18 | pandas: 2.3.1 | scikit-learn: 1.7.1 | joblib: 1.5.1


### C1. Audit berkas yang benar-benar tersedia

Jangan menebak lokasi model atau membuat data simulasi diam-diam. Kita perlu mengetahui ukuran berkas sumber dan apakah berkas metadata dapat dibaca.

In [2]:
required_files = {
    'data_bersih': CASE / 'data/hasil/data_bersih.csv',
    'metadata_06': RESULTS / 'laporan_praktikum06.json',
    'prediksi_06': RESULTS / 'prediksi_test.csv',
    'metrik_06': RESULTS / 'metrik_perbandingan.csv',
    'cv_06': RESULTS / 'ringkasan_cv.csv'
}
file_status = pd.DataFrame([{
    'berkas': key, 'tersedia': path.is_file(),
    'ukuran_byte': path.stat().st_size if path.is_file() else np.nan
} for key, path in required_files.items()])
display(file_status)
assert file_status['tersedia'].all(), 'Ada file wajib yang belum tersedia.'

,berkas,tersedia,ukuran_byte
0,data_bersih,True,28660
1,metadata_06,True,2206
2,prediksi_06,True,4361
3,metrik_06,True,303
4,cv_06,True,1291


## D. Membaca kontrak model dari Praktikum 06

*Model contract* menjelaskan jenis, jumlah, dan **urutan** kolom input, versi lingkungan, konfigurasi model, serta label positif. Skema input harus tetap stabil dari training sampai aplikasi Praktikum 08.

In [3]:
report_06 = json.loads((RESULTS / 'laporan_praktikum06.json').read_text(encoding='utf-8'))
FEATURES = list(report_06['features'])
TARGET = 'target_PE'
NUMERIC = ['USIA', 'PARITAS']
CATEGORICAL = [c for c in FEATURES if c not in NUMERIC]
params_06 = dict(report_06['best_params'])

assert len(FEATURES) == 8 and len(CATEGORICAL) == 6
assert report_06['positive_label'] == 1
assert all(k.startswith('classifier__') for k in params_06)

print('Modul asal:', report_06['module'])
print('8 prediktor berurutan:', FEATURES)
print('Best params dari TRAIN CV 06:', params_06)
print('Train/Test:', report_06['train_records'], '/', report_06['test_records'])
print('Test F1 PE (hasil 06):', report_06['test_metrics']['selected']['f1_PE'])

Modul asal: Praktikum 06 — Pipeline dan Evaluasi PE
8 prediktor berurutan: ['USIA', 'PARITAS', 'JARAK KELAHIRAN', 'RIW HIPERTENSI', 'OBESITAS', 'RIW DM', 'RIW HIPERTENSI/PE DALAM KELUARGA', 'SOSEK RENDAH']
Best params dari TRAIN CV 06: {'classifier__class_weight': 'balanced', 'classifier__max_depth': 2, 'classifier__min_samples_leaf': 5}
Train/Test: 420 / 141
Test F1 PE (hasil 06): 0.56


In [4]:
clean = pd.read_csv(CASE / 'data/hasil/data_bersih.csv')
assert set(FEATURES + [TARGET]).issubset(clean.columns)
X = clean[FEATURES].copy()
X[NUMERIC] = X[NUMERIC].apply(pd.to_numeric, errors='coerce')
for col in CATEGORICAL:
    X[col] = X[col].astype(object).where(X[col].notna(), np.nan)
y = clean[TARGET].astype(int)

assert len(X) == report_06['total_records'] == 561
assert int(y.sum()) == 41
assert set(y.unique()) == {0,1}
print('Shape fitur:',X.shape,'| positif PE:',int(y.sum()))
display(pd.DataFrame({'tipe':X.dtypes.astype(str), 'missing':X.isna().sum(),
                      'nilai_unik':X.nunique(dropna=True)}))

Shape fitur: (561, 8) | positif PE: 41


,tipe,missing,nilai_unik
USIA,float64,3,30
PARITAS,float64,0,7
JARAK KELAHIRAN,object,7,3
RIW HIPERTENSI,object,0,2
OBESITAS,object,0,2
RIW DM,object,0,2
RIW HIPERTENSI/PE DALAM KELUARGA,object,0,2
SOSEK RENDAH,object,0,2


## E. Reproduksi split tanpa membuka test untuk pemilihan model

Parameter `test_size=0.25`, `stratify=y`, dan `random_state=42` diambil dari metadata **Praktikum 06**. Ini menghasilkan train/test yang sama pada **urutan baris dataset yang sama**. Pada praktik produksi, pembagian data idealnya menyimpan identitas record stabil; di paket workshop ini validitas ID pasien unik belum tersedia.

In [5]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=report_06['split']['test_size'],
    random_state=report_06['split']['random_state'],
    stratify=y
)
assert len(X_train)==report_06['train_records']==420
assert len(X_test)==report_06['test_records']==141
assert int(y_test.sum())==report_06['class_test']['1']==10
assert set(X_train.index).isdisjoint(set(X_test.index))
print('Split identik menurut spesifikasi 06.')
display(pd.DataFrame({
    'subset':['train','test'],
    'n':[len(X_train),len(X_test)],
    'PE':[int(y_train.sum()),int(y_test.sum())]
}))

Split identik menurut spesifikasi 06.


,subset,n,PE
0,train,420,31
1,test,141,10


## F. Merekonstruksi pipeline final terpilih

Konfigurasi dicatat dalam metadata Praktikum 06, lalu digunakan sebagai `set_params`. **Tidak ada GridSearchCV baru**, karena tujuan modul ini bukan pemilihan model. Perhatikan nama langkah `'preprocess'` dan `'classifier'` yang konsisten dengan notebook 06.

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

def make_preprocessor():
    num = Pipeline([('imputer',SimpleImputer(strategy='median'))])
    cat = Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),
                    ('onehot',OneHotEncoder(handle_unknown='ignore',sparse_output=False))])
    return ColumnTransformer([
        ('num',num,NUMERIC),
        ('cat',cat,CATEGORICAL)
    ])

def make_tree():
    return Pipeline([
        ('preprocess',make_preprocessor()),
        ('classifier',DecisionTreeClassifier(
            max_depth=3,min_samples_leaf=10,
            class_weight='balanced', random_state=42))
    ])

final_model = make_tree().set_params(**params_06)
print(final_model)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['USIA', 'PARITAS']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['JARAK KELAHIRAN',
                                                   'RIW

In [7]:
# Training hanya pada train; jangan mem-fit ulang pada test.
final_model.fit(X_train, y_train)
print('Pipeline sudah fitted.')
print('Kelas:', final_model.classes_)
print('Jumlah fitur setelah OHE:',
      len(final_model.named_steps['preprocess'].get_feature_names_out()))

Pipeline sudah fitted.
Kelas: [0 1]
Jumlah fitur setelah OHE: 15


### F1. Periksa state imputasi dan encoding yang disimpan

Bukan hanya `max_depth`: median/modus, kategori encoder, dan pohon yang sudah dilatih harus ikut menjadi bagian model yang diekspor.

In [8]:
prep = final_model.named_steps['preprocess']
num_stats = prep.named_transformers_['num'].named_steps['imputer'].statistics_
cat_pipeline = prep.named_transformers_['cat']
cat_stats = cat_pipeline.named_steps['imputer'].statistics_
ohe = cat_pipeline.named_steps['onehot']
print('Median training:',dict(zip(NUMERIC,num_stats)))
display(pd.DataFrame({'feature':CATEGORICAL,'modus_train':cat_stats,
                      'jumlah_kategori':[len(c) for c in ohe.categories_]}))
print('Jumlah kolom output:',len(prep.get_feature_names_out()))

Median training: {'USIA': np.float64(28.0), 'PARITAS': np.float64(2.0)}


,feature,modus_train,jumlah_kategori
0,JARAK KELAHIRAN,anak pertama,3
1,RIW HIPERTENSI,Tidak,2
2,OBESITAS,Tidak,2
3,RIW DM,Tidak,2
4,RIW HIPERTENSI/PE DALAM KELUARGA,Tidak,2
5,SOSEK RENDAH,>UMR,2


Jumlah kolom output: 15


In [9]:
# Kontrak satu baris: fitur input mentah -> fitur ter-encode
idx = X_train.index[0]
one_raw = X_train.loc[[idx], FEATURES]
one_encoded = prep.transform(one_raw)
print('Shape input:', one_raw.shape, '| shape output:', one_encoded.shape)
display(one_raw)
display(pd.DataFrame(one_encoded,
    columns=prep.get_feature_names_out(),index=[idx]).T.head(20))

Shape input: (1, 8) | shape output: (1, 15)


,USIA,PARITAS,JARAK KELAHIRAN,RIW HIPERTENSI,OBESITAS,RIW DM,RIW HIPERTENSI/PE DALAM KELUARGA,SOSEK RENDAH
405,20.0,1.0,anak pertama,Tidak,Tidak,Tidak,Tidak,>UMR


,405
num__USIA,20.0
num__PARITAS,1.0
cat__JARAK KELAHIRAN_< 2 tahun,0.0
cat__JARAK KELAHIRAN_> 2 tahun,0.0
cat__JARAK KELAHIRAN_anak pertama,1.0
cat__RIW HIPERTENSI_Tidak,1.0
cat__RIW HIPERTENSI_Ya,0.0
cat__OBESITAS_Tidak,1.0
cat__OBESITAS_Ya,0.0
cat__RIW DM_Tidak,1.0


## G. Verifikasi terhadap hasil Praktikum 06

Ini tahap kontrol kualitas yang paling penting: model yang direkonstruksi harus menghasilkan **prediksi dan skor yang sama** pada holdout seperti artefak evaluasi Modul 06, sebelum kita menyebutnya sebagai model yang sama. Perbandingan ini bersifat **audit reproduksi**, bukan proses memilih model/threshold baru dari data test.

In [10]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

def binary_metrics(y_true,y_pred):
    tn,fp,fn,tp = confusion_matrix(y_true,y_pred,labels=[0,1]).ravel()
    return dict(accuracy=float(accuracy_score(y_true,y_pred)),
                precision_PE=float(precision_score(y_true,y_pred,zero_division=0)),
                recall_PE=float(recall_score(y_true,y_pred,zero_division=0)),
                f1_PE=float(f1_score(y_true,y_pred,zero_division=0)),
                specificity=float(tn/(tn+fp)) if tn+fp else None,
                TN=int(tn),FP=int(fp),FN=int(fn),TP=int(tp))

pred_before = final_model.predict(X_test)
score_before = final_model.predict_proba(X_test)[:, list(final_model.classes_).index(1)]
result_now = binary_metrics(y_test,pred_before)
print('Metrik rekonstruksi:')
display(pd.DataFrame([result_now]))

Metrik rekonstruksi:


,accuracy,precision_PE,recall_PE,f1_PE,specificity,TN,FP,FN,TP
0,0.921986,0.466667,0.7,0.56,0.938931,123,8,3,7


In [11]:
# Periksa kesetaraan hasil dengan hasil test terpilih di Modul 06.
expected_metrics = report_06['test_metrics']['selected']
for key in ['accuracy','precision_PE','recall_PE','f1_PE','specificity']:
    assert np.isclose(result_now[key],expected_metrics[key],atol=1e-12),key
for key in ['TN','FP','FN','TP']:
    assert result_now[key] == expected_metrics[key],key

saved_06 = pd.read_csv(RESULTS/'prediksi_test.csv')
assert len(saved_06)==len(X_test)
assert np.array_equal(saved_06['aktual'].to_numpy(), y_test.to_numpy())
assert np.array_equal(saved_06['prediksi'].to_numpy(), pred_before)
assert np.allclose(saved_06['skor_model_PE'].to_numpy(),score_before,atol=1e-12)
print('VALID: 141 prediksi, skor, dan confusion matrix cocok dengan hasil 06.')

VALID: 141 prediksi, skor, dan confusion matrix cocok dengan hasil 06.


### G1. Membedakan model sumber dan model terpilih

Paket asal menyediakan `pe_workflow.train_export()` untuk **pohon konfigurasi tetap** (`max_depth=3`, `min_samples_leaf=10`). Praktikum 06 yang telah direvisi juga mengevaluasi konfigurasi tetap tersebut, tetapi **model finalnya dipilih melalui train CV** (`max_depth=2`, `min_samples_leaf=5`, `class_weight=balanced`). Memanggil `train_export()` di sini justru akan menimpa nama artefak model final dengan model berbeda.

**Karena itu, notebook ini tidak memanggil `train_export()`**. Seluruh metadata menyebut dengan jelas bahwa artefak `.joblib` adalah model hasil CV Praktikum 06.

In [12]:
comparison_06 = pd.read_csv(RESULTS/'metrik_perbandingan.csv')
display(comparison_06[['model','accuracy','recall_PE','f1_PE','TN','FP','FN','TP']])
print('Model yang akan diekspor:', comparison_06.iloc[-1]['model'])
assert comparison_06.iloc[-1]['model']=='Pohon terpilih via CV'

,model,accuracy,recall_PE,f1_PE,TN,FP,FN,TP
0,Baseline mayoritas,0.929078,0.0,0.000000,131,0,10,0
1,Pohon sumber (fixed),0.822695,0.7,0.358974,109,22,3,7
2,Pohon terpilih via CV,0.921986,0.7,0.560000,123,8,3,7


Model yang akan diekspor: Pohon terpilih via CV


## H. Ekspor pipeline final dengan joblib

Simpan satu file yang memuat preprocessing **dan** classifier. Berkas tambahan untuk laporan, metadata, atau contoh input disimpan terpisah agar dapat dibaca tanpa memuat objek Python.

In [13]:
MODEL_NAME = 'pe_workshop_pipeline.joblib'
model_path = MODEL_DIR/MODEL_NAME
joblib.dump(final_model, model_path, compress=3)
assert model_path.exists() and model_path.stat().st_size > 0
print('Model disimpan:',model_path)
print('Ukuran:',model_path.stat().st_size,'byte')

Model disimpan: /Users/masaboe/Documents/NGAJAR/ngudiWaluyo2026/Materi/Workshop_From_Data_to_ML/studi_kasus/model/pe_workshop_pipeline.joblib
Ukuran: 2275 byte


### H1. Reload: hasil harus identik

Jalankan `joblib.load` **hanya terhadap berkas yang baru dibuat sendiri**. Perbandingan label harus sama persis; skor float dicek dengan toleransi numerik ketat.

In [14]:
loaded_model = joblib.load(model_path)  # artefak ini dibuat oleh sel sebelumnya
pred_after = loaded_model.predict(X_test)
score_after = loaded_model.predict_proba(X_test)[:,list(loaded_model.classes_).index(1)]
assert np.array_equal(pred_before,pred_after)
assert np.allclose(score_before,score_after,rtol=0,atol=1e-12)
print('PASS: 141 label dan skor konsisten sebelum/sesudah reload.')

PASS: 141 label dan skor konsisten sebelum/sesudah reload.


## I. Pemeriksaan kontrak fitur saat inferensi

Aplikasi wajib mengirim kedelapan nama fitur. Urutan kolom pada DataFrame idealnya mengikuti kontrak training, meskipun `ColumnTransformer` memilih kolom berdasarkan nama. Kolom tidak dikenal tidak boleh diam-diam dijadikan prediktor.

In [15]:
# Data contoh eksplisit sintetis: BUKAN data pasien atau contoh dari workbook asal.
synthetic_rows = [
    {'USIA':24,'PARITAS':0,'JARAK KELAHIRAN':'anak pertama',
     'RIW HIPERTENSI':'Tidak','OBESITAS':'Tidak','RIW DM':'Tidak',
     'RIW HIPERTENSI/PE DALAM KELUARGA':'Tidak','SOSEK RENDAH':'>UMR'},
    {'USIA':35,'PARITAS':2,'JARAK KELAHIRAN':'> 2 tahun',
     'RIW HIPERTENSI':'Ya','OBESITAS':'Ya','RIW DM':'Tidak',
     'RIW HIPERTENSI/PE DALAM KELUARGA':'Ada','SOSEK RENDAH':'<UMR'},
    {'USIA':29,'PARITAS':1,'JARAK KELAHIRAN':'< 2 tahun',
     'RIW HIPERTENSI':'Tidak','OBESITAS':'Ya','RIW DM':'Tidak',
     'RIW HIPERTENSI/PE DALAM KELUARGA':'Tidak','SOSEK RENDAH':'>UMR'}
]
examples = pd.DataFrame(synthetic_rows)[FEATURES]
examples.to_csv(DEPLOY_DATA/'contoh_input.csv',index=False)
example_pred = loaded_model.predict(examples)
example_score = loaded_model.predict_proba(examples)[:,list(loaded_model.classes_).index(1)]
view = examples.copy()
view['prediksi_label_PE'] = example_pred
view['skor_model_PE_bukan_risiko_klinis'] = example_score
display(view)

,USIA,PARITAS,JARAK KELAHIRAN,RIW HIPERTENSI,OBESITAS,RIW DM,RIW HIPERTENSI/PE DALAM KELUARGA,SOSEK RENDAH,prediksi_label_PE,skor_model_PE_bukan_risiko_klinis
0,24,0,anak pertama,Tidak,Tidak,Tidak,Tidak,>UMR,0,0.255837
1,35,2,> 2 tahun,Ya,Ya,Tidak,Ada,<UMR,1,0.930004
2,29,1,< 2 tahun,Tidak,Ya,Tidak,Tidak,>UMR,0,0.255837


In [16]:
def validate_input(frame, feature_names=FEATURES):
    if not isinstance(frame,pd.DataFrame):
        raise TypeError('Input harus pandas.DataFrame.')
    missing = [c for c in feature_names if c not in frame.columns]
    if missing:
        raise ValueError(f'Fitur wajib hilang: {missing}')
    return frame.loc[:, feature_names].copy()

check = validate_input(examples)
assert list(check.columns)==FEATURES
assert np.array_equal(loaded_model.predict(check),example_pred)
try:
    validate_input(examples.drop(columns=['USIA']))
except ValueError as ex:
    print('VALIDASI BERHASIL menolak input tanpa kolom USIA:',ex)
else:
    raise AssertionError('Input tanpa kolom wajib semestinya ditolak.')

VALIDASI BERHASIL menolak input tanpa kolom USIA: Fitur wajib hilang: ['USIA']


### I1. *Unknown category* dan missing value

`OneHotEncoder(handle_unknown='ignore')` dapat memproses kategori baru dengan vektor nol pada blok kategori tersebut. Ini **menghindari error**, tetapi tidak berarti kategori baru tersebut dipahami model. `SimpleImputer` yang telah *fit* menangani nilai numerik/kategori kosong sesuai statistik training.

In [17]:
probe = examples.iloc[[0]].copy()
probe['USIA'] = np.nan
probe['RIW HIPERTENSI'] = 'kategori tidak dikenal'
probe_prediction = loaded_model.predict(probe)[0]
probe_score = loaded_model.predict_proba(probe)[0,list(loaded_model.classes_).index(1)]
assert np.isfinite(probe_score)
print('Pipeline dapat menerima NaN dan kategori baru:',int(probe_prediction),
      '| skor:',round(float(probe_score),4))
print('Catatan: kategori baru memerlukan review data, bukan otomatis dianggap valid.')

Pipeline dapat menerima NaN dan kategori baru: 0 | skor: 0.2558
Catatan: kategori baru memerlukan review data, bukan otomatis dianggap valid.


## J. Model card dan informasi reproduksi

*Model card* berfungsi untuk dokumentasi: tujuan, sumber data, kelas, daftar fitur, parameter, metrik, dependensi, dan batas penggunaan. **Model card tidak menjadikan model layak klinis**; justru membantu menyatakan keterbatasan secara eksplisit.

In [18]:
# Dokumentasi ringkas, konsisten dengan report Praktikum 06.
model_card = {
    'title':'Decision Tree untuk demonstrasi klasifikasi label PE',
    'module':'Praktikum 07 — Ekspor Model dan Hasil',
    'origin_module':'Praktikum 06 — model terpilih via CV train',
    'purpose':'Pendidikan dan demonstrasi pipeline; tidak untuk keputusan klinis',
    'model_type':'Pipeline(ColumnTransformer, DecisionTreeClassifier)',
    'model_filename':MODEL_NAME,
    'sklearn_version':sklearn.__version__,
    'python_version':platform.python_version(),
    'pandas_version':pd.__version__,
    'joblib_version':joblib.__version__,
    'input_features':FEATURES,
    'features':FEATURES,
    'excluded_features':['NO','NAMA','Unnamed: 12','RIW PE'],
    'target':'target_PE',
    'label_mapping':{'0':'Non PE','1':'PE / kelompok PE menurut label latihan'},
    'train_records':int(len(X_train)),
    'test_records':int(len(X_test)),
    'n_test_positive':int(y_test.sum()),
    'split_config':report_06['split'],
    'cv_config':report_06['cv'],
    'model_selection':'Best parameters selected in Praktikum 06 via TRAIN 5-fold CV, scoring F1 PE',
    'hyperparameters':params_06,
    'best_train_cv_f1_PE':report_06['best_cv_f1_PE'],
    'test_metrics':report_06['test_metrics']['selected'],
    'predicted_score_notice':'Keluaran predict_proba belum dikalibrasi; bukan probabilitas risiko klinis',
    'limitations':report_06['limitations'] + [
        'Data test hanya memiliki 10 baris positif',
        'Kategori baru yang diabaikan encoder belum tentu sesuai populasi training',
        'Model tidak divalidasi untuk pasien nyata, situs eksternal, atau perubahan waktu'
    ],
    'safety':'Tidak untuk diagnosis, keputusan medis, atau penggunaan klinis',
    'source_report':'studi_kasus/hasil/laporan_praktikum06.json'
}
card_path = MODEL_DIR/'model_card.json'
card_path.write_text(json.dumps(model_card,ensure_ascii=False,indent=2),encoding='utf-8')
card_read = json.loads(card_path.read_text(encoding='utf-8'))
assert card_read['sklearn_version']==sklearn.__version__
assert card_read['input_features']==FEATURES
print('Model card tersimpan:',card_path)
display(pd.DataFrame([{
    'Model asal':card_read['origin_module'],
    'F1 test PE':card_read['test_metrics']['f1_PE'],
    'Jumlah positif test':card_read['n_test_positive']
}]))

Model card tersimpan: /Users/masaboe/Documents/NGAJAR/ngudiWaluyo2026/Materi/Workshop_From_Data_to_ML/studi_kasus/model/model_card.json


,Model asal,F1 test PE,Jumlah positif test
0,Praktikum 06 — model terpilih via CV train,0.56,10


## K. Hash, manifest, dan salinan deployment

Satu file pipeline dapat memiliki dua lokasi yang sama persis: folder eksperimen dan folder aplikasi. **Salin** file terlatih, jangan melatih model ulang secara tidak sengaja pada lokasi deployment. Hash membantu menguji kesamaan byte hasil salin.

In [19]:
def sha256_file(path, chunk_bytes=1024*1024):
    sha=hashlib.sha256()
    with open(path,'rb') as f:
        for buf in iter(lambda:f.read(chunk_bytes),b''):
            sha.update(buf)
    return sha.hexdigest()

deploy_model=DEPLOY_MODELS/MODEL_NAME
deploy_card=DEPLOY_MODELS/'model_card.json'
shutil.copy2(model_path,deploy_model)
shutil.copy2(card_path,deploy_card)
assert model_path.read_bytes()==deploy_model.read_bytes()
assert sha256_file(model_path)==sha256_file(deploy_model)
assert sha256_file(card_path)==sha256_file(deploy_card)
print('SHA-256 model:',sha256_file(model_path))
print('Deployment copy:',deploy_model)

SHA-256 model: 90c198ba9d5a8a0a23c2c67d7f3d02d64f3801bb25f63549db36379f9e0e16ad
Deployment copy: /Users/masaboe/Documents/NGAJAR/ngudiWaluyo2026/Materi/Workshop_From_Data_to_ML/deployment/versi_2_hasil_studi_kasus/models/pe_workshop_pipeline.joblib


In [20]:
# File manifest dengan hash artefak hasil akhir.
manifest_paths = [
    model_path, card_path, deploy_model, deploy_card,
    DEPLOY_DATA/'contoh_input.csv',
    RESULTS/'laporan_praktikum06.json',
    RESULTS/'prediksi_test.csv'
]
manifest = pd.DataFrame([{
    'path':str(p.relative_to(ROOT)),
    'bytes':p.stat().st_size,
    'sha256':sha256_file(p)
} for p in manifest_paths])
manifest_path=RESULTS/'manifest_praktikum07.csv'
manifest.to_csv(manifest_path,index=False)
display(manifest)

,path,bytes,sha256
0,studi_kasus/model/pe_workshop_pipeline.joblib,2275,90c198ba9d5a8a0a23c2c67d7f3d02d64f3801bb25f635...
1,studi_kasus/model/model_card.json,2756,565bdd85b8c4eaf3c3381472478924940753d68a0184fa...
2,deployment/versi_2_hasil_studi_kasus/models/pe...,2275,90c198ba9d5a8a0a23c2c67d7f3d02d64f3801bb25f635...
3,deployment/versi_2_hasil_studi_kasus/models/mo...,2756,565bdd85b8c4eaf3c3381472478924940753d68a0184fa...
4,deployment/versi_2_hasil_studi_kasus/data/cont...,230,7ebca0defcf847631f08478359971f29df5d433dcdb70d...
5,studi_kasus/hasil/laporan_praktikum06.json,2206,fd6fbf5dc6c7438deee16fac6d36faa910d1fd04787a20...
6,studi_kasus/hasil/prediksi_test.csv,4361,912cbefcf020abe959f9d08b3fa566d96981edcfc0f5d4...


### K1. Simulasi load dari folder deployment

Praktikum 08 akan membaca file model di folder deployment. Periksa bahwa nama fitur dan kelas prediksinya tetap sama, lalu **jangan** menggunakan metrik model sumber yang berbeda.

In [21]:
app_model=joblib.load(deploy_model)  # disalin dari objek yang baru dibuat sendiri
app_example=pd.read_csv(DEPLOY_DATA/'contoh_input.csv')
app_input=validate_input(app_example)
assert np.array_equal(app_model.predict(app_input),example_pred)
assert np.allclose(app_model.predict_proba(app_input),
                   loaded_model.predict_proba(app_input),rtol=0,atol=1e-12)
app_card=json.loads(deploy_card.read_text(encoding='utf-8'))
assert app_card['sklearn_version']==sklearn.__version__
print('PASS: model dan model card bisa dipakai folder deployment Praktikum 08.')

PASS: model dan model card bisa dipakai folder deployment Praktikum 08.


## L. Ekspor laporan prediksi dan aturan model

Berkas hasil yang disimpan pada modul ini disertai penanda **`07`**, sehingga tidak menimpa hasil evaluasi Modul 06. Baris sumber bersifat penanda posisi dalam dataset latihan, **bukan ID pasien**.

In [22]:
from sklearn.tree import export_text
pred_table = pd.DataFrame({
    'baris_sumber_perkiraan':X_test.index.to_numpy()+2,
    'aktual':y_test.to_numpy(),
    'prediksi':pred_after,
    'skor_model_PE':score_after
})
pred_table.to_csv(RESULTS/'prediksi_test_07.csv',index=False)
pd.DataFrame([result_now],index=['decision_tree_cv_selected']).to_csv(
    RESULTS/'metrik_07.csv',index_label='model')
feature_names=loaded_model.named_steps['preprocess'].get_feature_names_out()
text_rules=export_text(loaded_model.named_steps['classifier'],feature_names=list(feature_names))
(RESULTS/'aturan_pohon_07.txt').write_text(text_rules,encoding='utf-8')
print('Aturan pohon final (mekanisme teknis, bukan aturan medis):')
print(text_rules[:1500])

Aturan pohon final (mekanisme teknis, bukan aturan medis):
|--- cat__RIW HIPERTENSI_Ya <= 0.50
|   |--- cat__RIW HIPERTENSI/PE DALAM KELUARGA_Ada <= 0.50
|   |   |--- class: 0
|   |--- cat__RIW HIPERTENSI/PE DALAM KELUARGA_Ada >  0.50
|   |   |--- class: 1
|--- cat__RIW HIPERTENSI_Ya >  0.50
|   |--- num__USIA <= 35.50
|   |   |--- class: 1
|   |--- num__USIA >  35.50
|   |   |--- class: 0



In [23]:
# Checkpoint ketuntasan: semua file siap dan prediksi sesuai.
assert (RESULTS/'prediksi_test_07.csv').is_file()
assert len(pd.read_csv(RESULTS/'prediksi_test_07.csv'))==len(y_test)
assert pd.read_csv(RESULTS/'prediksi_test_07.csv')['prediksi'].equals(
    pd.read_csv(RESULTS/'prediksi_test.csv')['prediksi'])
assert len(json.loads(card_path.read_text(encoding='utf-8'))['input_features'])==8
assert sha256_file(model_path)==sha256_file(deploy_model)
print('PASS: seluruh pemeriksaan dasar Praktikum 07 berhasil.')

PASS: seluruh pemeriksaan dasar Praktikum 07 berhasil.


## M. Diagram alur ekspor dan inferensi

```text
Praktikum 05: data_bersih.csv
            │
            ▼
Praktikum 06: split + CV pada train → best_params + prediksi test
            │
            ▼
Praktikum 07: reconstruct pipeline → fit train → audit vs hasil 06
            │
            ├── pipeline.joblib → reload → compare predict
            ├── model_card.json + manifest hash
            ├── laporan evaluasi dan aturan pohon
            └── salinan identik ke deployment/versi_2_hasil_studi_kasus/
                                │
                                ▼
                 Praktikum 08: inference + Streamlit
```

**Refleksi:** Apa yang harus dilakukan jika skor hasil *reload* berbeda, versi library tidak kompatibel, atau fitur input hilang? **Hentikan inferensi**, audit perubahan data/pipeline/environment, dan regenerasi artefak dari sumber yang tepercaya.

## N. Latihan konseptual (12 soal)

Jawab terlebih dahulu sebelum membuka pembahasan pada bagian Q.

1. Mengapa menyimpan `DecisionTreeClassifier` saja tidak cukup untuk aplikasi Praktikum 08?
2. Apa beda `fit`, `transform`, dan `predict` dalam rangkaian pipeline?
3. Apa fungsi `joblib.dump` dan `joblib.load`?
4. Mengapa model hasil CV Praktikum 06 **tidak boleh** digantikan diam-diam oleh `train_export()` dari paket sumber?
5. Mengapa `best_params` dibaca dari metadata, bukan dipilih ulang dari test?
6. Apa perbedaan metadata `model_card.json` dengan file `.joblib`?
7. Mengapa SHA-256 bukan bukti bahwa file `.joblib` aman untuk dimuat?
8. Apa arti hasil `np.array_equal` pada prediksi sebelum/sesudah reload?
9. Jika berkas model dibuat pada scikit-learn versi berbeda, tindakan apa yang tepat?
10. Mengapa delapan nama dan urutan fitur perlu didokumentasikan?
11. Mengapa skor `predict_proba` tidak boleh langsung disebut risiko PE individual?
12. Sebutkan tiga keterbatasan studi kasus yang wajib tercantum di model card.

## O. Latihan pemrograman (10 soal)

1. Tampilkan lima fitur pertama dari kontrak input model.
2. Periksa jumlah baris positif di train dan test dari split yang dihasilkan.
3. Tampilkan median imputasi untuk `USIA` dan `PARITAS`.
4. Hitung jumlah kategori hasil One-Hot Encoding dari enam fitur kategorikal.
5. Lakukan prediksi pada satu record sintetis pertama.
6. Bandingkan skor untuk contoh sintetis pertama antara model memori dan model reload.
7. Cetak ukuran berkas `.joblib` dalam KiB.
8. Periksa SHA-256 dua salinan model.
9. Uji bahwa validasi menolak DataFrame tanpa fitur `PARITAS`.
10. Baca ulang `model_card.json` dan tampilkan skor F1 PE dari Modul 06.

## P. Tantangan terapan (3 soal)

**T1. Audit kesalahan deployment.** Buat DataFrame berisi nama, ukuran, dan hash setiap file model/card pada folder eksperimen dan deployment; tentukan apakah salinan identik.

**T2. Audit reproduksi.** Bandingkan *semua* 141 skor model dari CSV Modul 06 dengan hasil prediksi setelah reload; laporkan selisih absolut terbesar.

**T3. Kontrak inferensi.** Buat fungsi yang menerima DataFrame, memeriksa 8 fitur wajib, mengembalikan `kelas_prediksi` dan `skor_PE`, dan menolak fitur wajib yang hilang. Jangan menginterpretasikan skor sebagai risiko klinis.

## Q. Kunci jawaban konseptual dan pembahasan

1. **Preprocessing ikut dipelajari saat fit:** classifier mengharapkan fitur hasil imputasi dan encoding yang konsisten.
2. `fit` mempelajari state dari train; `transform` mengubah input dengan state itu; `predict` menghasilkan label menggunakan pipeline/model fitted.
3. `dump` menulis artefak Python; `load` membaca dan memulihkan objek—hanya dari sumber tepercaya.
4. `train_export` sumber melatih **pohon fixed**, bukan model final hasil seleksi CV pada Praktikum 06. Menimpanya membuat klaim evaluasi tidak sesuai artefak.
5. Test harus tetap independen dari pemilihan model. Modul 07 menguji reproduksi model terpilih, bukan tuning ulang.
6. `.joblib` berisi objek fitted; `model_card.json` berisi deskripsi, kontrak, metrik, versi, dan keterbatasan.
7. Hash hanya memeriksa apakah bytes sama/berubah, tidak menilai keamanan isi berkas.
8. Label prediksi sebelum/sesudah reload konsisten; skor juga perlu dicek dengan toleransi float.
9. Cocokkan environment tepercaya, ulangi verifikasi, dan jika perlu latih ulang pipeline dengan versi yang telah diaudit; jangan mengabaikan peringatan inkompatibilitas.
10. Agar input baru memetakan nilai ke transformasi/kolom benar dan antarmuka bisa divalidasi.
11. Pohon belum dikalibrasi, validasi klinis tidak ada, dan `class_weight` mengubah pembobotan training.
12. Contoh: split baris bukan split pasien terverifikasi, hanya 10 positif di test, timing fitur/outcome belum diaudit, tidak ada validasi eksternal, dan skor tidak terkalibrasi.

## R. Kunci jawaban kode (10 latihan)

Sel-sel berikut merupakan solusi yang dapat dieksekusi. Perhatikan bahwa beberapa pengujian menghasilkan pesan pengecualian yang **sengaja ditangani** sebagai bagian pembelajaran.

In [24]:
# R1. Lima fitur pertama
print(FEATURES[:5])

['USIA', 'PARITAS', 'JARAK KELAHIRAN', 'RIW HIPERTENSI', 'OBESITAS']


In [25]:
# R2. Jumlah sampel positif pada split
print('PE train:',int(y_train.sum()),'| PE test:',int(y_test.sum()))

PE train: 31 | PE test: 10


In [26]:
# R3. Median train yang dipakai fitted imputer
display(pd.DataFrame({'kolom':NUMERIC,'median_training':num_stats}))

,kolom,median_training
0,USIA,28.0
1,PARITAS,2.0


In [27]:
# R4. Jumlah kategori One-Hot Encoder
sizes=[len(x) for x in ohe.categories_]
print('Total kategori OHE:',sum(sizes),'| rincian:',dict(zip(CATEGORICAL,sizes)))

Total kategori OHE: 13 | rincian: {'JARAK KELAHIRAN': 3, 'RIW HIPERTENSI': 2, 'OBESITAS': 2, 'RIW DM': 2, 'RIW HIPERTENSI/PE DALAM KELUARGA': 2, 'SOSEK RENDAH': 2}


In [28]:
# R5. Prediksi satu contoh sintetis
print('Prediksi baris 0:',int(loaded_model.predict(examples.iloc[[0]])[0]))

Prediksi baris 0: 0


In [29]:
# R6. Bandingkan probabilitas output kedua objek model
s1=final_model.predict_proba(examples.iloc[[0]])
s2=loaded_model.predict_proba(examples.iloc[[0]])
assert np.allclose(s1,s2,atol=1e-12)
print('Konsisten:',s1.ravel())

Konsisten: [0.7441631 0.2558369]


In [30]:
# R7. Ukuran artefak model dalam KiB
print('Ukuran model: %.2f KiB' % (model_path.stat().st_size/1024))

Ukuran model: 2.22 KiB


In [31]:
# R8. SHA-256 salinan
display(pd.DataFrame({'lokasi':['eksperimen','deployment'],
                      'sha256':[sha256_file(model_path),sha256_file(deploy_model)]}))

,lokasi,sha256
0,eksperimen,90c198ba9d5a8a0a23c2c67d7f3d02d64f3801bb25f635...
1,deployment,90c198ba9d5a8a0a23c2c67d7f3d02d64f3801bb25f635...


In [32]:
# R9. Kolom PARITAS hilang -> validasi menolak
try:
    validate_input(examples.drop(columns='PARITAS'))
except ValueError as exc:
    print('PASS, masukan ditolak:',exc)

PASS, masukan ditolak: Fitur wajib hilang: ['PARITAS']


In [33]:
# R10. F1 positif dari model card
card_check=json.loads(card_path.read_text(encoding='utf-8'))
print('F1 kelas PE pada test:',card_check['test_metrics']['f1_PE'])

F1 kelas PE pada test: 0.56


## S. Kunci jawaban tantangan terapan

Solusi memakai berkas yang benar-benar telah dibuat dari langkah ekspor, bukan berkas fiktif. Nilai hash bergantung pada byte yang disimpan dan mungkin berbeda antar lingkungan.

In [34]:
# S1. Tabel integritas: ukuran dan hash
check_paths=[model_path,deploy_model,card_path,deploy_card]
audit=pd.DataFrame([{'lokasi':str(p.relative_to(ROOT)),
                     'ukuran_byte':p.stat().st_size,
                     'hash':sha256_file(p)} for p in check_paths])
display(audit)
assert audit.iloc[0]['hash']==audit.iloc[1]['hash']
assert audit.iloc[2]['hash']==audit.iloc[3]['hash']

,lokasi,ukuran_byte,hash
0,studi_kasus/model/pe_workshop_pipeline.joblib,2275,90c198ba9d5a8a0a23c2c67d7f3d02d64f3801bb25f635...
1,deployment/versi_2_hasil_studi_kasus/models/pe...,2275,90c198ba9d5a8a0a23c2c67d7f3d02d64f3801bb25f635...
2,studi_kasus/model/model_card.json,2756,565bdd85b8c4eaf3c3381472478924940753d68a0184fa...
3,deployment/versi_2_hasil_studi_kasus/models/mo...,2756,565bdd85b8c4eaf3c3381472478924940753d68a0184fa...


In [35]:
# S2. Deviasi skor hasil reload dibanding output 06
saved_scores=pd.read_csv(RESULTS/'prediksi_test.csv')['skor_model_PE'].to_numpy()
max_delta=float(np.max(np.abs(saved_scores-score_after)))
print('Maksimum selisih absolut skor dari 141 baris:',max_delta)
assert max_delta < 1e-12

Maksimum selisih absolut skor dari 141 baris: 5.551115123125783e-17


In [36]:
# S3. Inferensi dengan pemeriksaan kontrak input
def predict_with_contract(model, frame):
    verified=validate_input(frame)
    positive_idx=list(model.classes_).index(1)
    return pd.DataFrame({
        'kelas_prediksi':model.predict(verified).astype(int),
        'skor_PE':model.predict_proba(verified)[:,positive_idx]
    },index=verified.index)

inference_demo=predict_with_contract(app_model,examples)
display(inference_demo)
try:
    predict_with_contract(app_model,examples.drop(columns=['OBESITAS']))
except ValueError as exc:
    print('PASS: error fitur hilang terdeteksi:',exc)

,kelas_prediksi,skor_PE
0,0,0.255837
1,1,0.930004
2,0,0.255837


PASS: error fitur hilang terdeteksi: Fitur wajib hilang: ['OBESITAS']


## T. Rangkuman dan persiapan Praktikum 08

**Yang sudah dipelajari:**

- Pipeline fitted yang disimpan memuat state preprocessing dan model sekaligus.
- Model final adalah **hasil pemilihan CV TRAIN Praktikum 06**; tidak diganti dengan pohon fixed.
- Label dan skor pada 141 test cocok dengan hasil evaluasi 06, serta tetap sama sesudah reload.
- Dua berkas `.joblib` di folder eksperimen dan deployment identik berdasarkan hash; metadata menjelaskan versi, input, tujuan, dan keterbatasan.
- Contoh input bersifat **sintetis**, tanpa informasi identitas pasien.

**Untuk Praktikum 08:** gunakan `deployment/versi_2_hasil_studi_kasus/models/pe_workshop_pipeline.joblib`, `model_card.json`, dan `data/contoh_input.csv`. Jalankan Streamlit **hanya untuk demonstrasi pendidikan**, bukan untuk penggunaan klinis. Jika lingkungan berbeda, verifikasi kompatibilitas library dan regresi prediksi.

**Catatan versi:** model lama dari paket awal berbeda dengan model final CV pada paket revisi ini. Hindari mencampur file `.joblib` dan laporan dari kedua versi.